In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)
from sklearn.model_selection import RandomizedSearchCV
from lightgbm import LGBMClassifier
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import os
import joblib

In [ ]:
# Loading the datasets

df = pd.read_csv("../Data/Cleaned/ChargingRecords_Feature_Engineered.csv")

# Calculate median demand
demand_threshold = df['Demand'].median()

# Create classfication target
df['DemandCategory']= np.where(df['Demand']>= demand_threshold, 1,0)

# Display threshold and distribution

print("Median Threshold: " , demand_threshold)

print("\nDistribution of Demand Categories: ")

print(df["DemandCategory"]
    .value_counts()
    .rename(index={0: "Normal Demand", 1: "High Demand"}))

print("\nPercentage Distribution:")

print(
    df["DemandCategory"]
    .value_counts(normalize=True)
    .mul(100)
    .rename(index={0: "Normal Demand", 1: "High Demand"})
    .round(2)
)

print("\nDataset Shape:", df.shape)

In [ ]:
# Separate features and target
X = df.drop(columns=["Demand", "DemandCategory"])
y = df["DemandCategory"]

# Train-test split with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Display shapes
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

print("\nTraining Target Distribution:")
print(y_train.value_counts())

print("\nTesting Target Distribution:")
print(y_test.value_counts())

In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=1000,
            random_state=42
        ))
    ]),

    "Decision Tree": DecisionTreeClassifier(
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ),

    "KNN": Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier(
            n_neighbors=5,
            n_jobs=-1
        ))
    ]),
    
     "SVM" :Pipeline([
            ("scaler" , StandardScaler()),
            ("model", LinearSVC(
                max_iter=5000,
                random_state=42
            ))
        ]),
        
     "XMBoost":XGBClassifier(
            n_estimators = 100,
            learning_rate = 0.05,
            max_depth = 6,
            random_state = 42,
            n_jobs = -1,
            eval_metric = 'logloss'
            ),
        
     "LightGBM":LGBMClassifier(
            n_estimators=100,
            learning_rate=0.05,
            max_depth=6,
            random_state=42,
            n_jobs=-1,
            verbosity = -1
        )    
}

print("Models initialized successfully!")

for name in models:
    print(name)

In [ ]:
for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)

print("\nAll four classification models trained successfully!")

In [ ]:
# Evaluate classification models

results = []

for name , model in models.items():
    # Prediction
    y_pred = model.predict(X_test)
    
    # Get score for ROC-AUC
    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_test)[:, 1]
    else:
        y_score = model.decision_function(X_test)
    
    results.append({
        "Models":name,
        "Accuracy":accuracy_score(y_test,y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1-Score": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_score)
    })
    
classification_results = pd.DataFrame(results)

classification_results = classification_results.sort_values(
    by="F1-Score", 
    ascending=False
).reset_index(drop=True)

classification_results.round(4)

In [ ]:
# Tune the LightGBM

lgbm = LGBMClassifier(
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

param_grid = {
    "n_estimators": [100, 200, 300, 500],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "max_depth": [-1, 5, 7, 10],
    "num_leaves": [15, 31, 50, 70],
    "subsample": [0.8, 0.9, 1.0],
    "colsample_bytree": [0.8, 0.9, 1.0]
}

random_search = RandomizedSearchCV(
    estimator=lgbm,
    param_distributions=param_grid,
    n_iter=20,
    scoring="f1",
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1
)


random_search.fit(X_train, y_train)

best_lgbm_model = random_search.best_estimator_

print("Best Parameters:", random_search.best_params_)
print("Best CV F1-score:", random_search.best_score_)

In [ ]:
# Predictions
y_pred_tuned = best_lgbm_model.predict(X_test)
y_prob_tuned = best_lgbm_model.predict_proba(X_test)[:, 1]

# Evaluation
tuned_results = {
    "Accuracy": accuracy_score(y_test, y_pred_tuned),
    "Precision": precision_score(y_test, y_pred_tuned),
    "Recall": recall_score(y_test, y_pred_tuned),
    "F1-Score": f1_score(y_test, y_pred_tuned),
    "ROC-AUC": roc_auc_score(y_test, y_prob_tuned)
}

print("Tuned LightGBM Results:")
for metric, score in tuned_results.items():
    print(f"{metric}: {score:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred_tuned,
        target_names=["Normal Demand", "High Demand"]
    )
)

In [ ]:
cm = confusion_matrix(y_test, y_pred_tuned)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Normal Demand", "High Demand"]
)

disp.plot(cmap="Blues", values_format="d")
plt.title("Tuned LightGBM - Confusion Matrix")
plt.show()

In [ ]:
feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": best_lgbm_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

# Display top 15 features
top_features = feature_importance.head(15)

plt.figure(figsize=(10, 7))

sns.barplot(
    data=top_features,
    x="Importance",
    y="Feature"
)

plt.title("Top 15 Feature Importances - Tuned LightGBM")
plt.xlabel("Importance")
plt.ylabel("Features")
plt.tight_layout()
plt.show()

display(top_features)

In [ ]:
# Create directories
os.makedirs("../Models", exist_ok=True)
os.makedirs("../Reports", exist_ok=True)

# Save best model
joblib.dump(
    best_lgbm_model,
    "../Models/best_classification_model.pkl"
)

# Save model comparison
classification_results.to_csv(
    "../Reports/classification_results.csv",
    index=False
)

# Save tuned model predictions
prediction_results = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": y_pred_tuned,
    "Probability_High_Demand": y_prob_tuned
})

prediction_results.to_csv(
    "../Reports/classification_predictions.csv",
    index=False
)

# Save feature importance
feature_importance.to_csv(
    "../Reports/classification_feature_importance.csv",
    index=False
)

print("Classification model and reports saved successfully!")